In [0]:
select * from gizmobox.bronze.v_customers;

Databricks data profile. Run in Databricks to view.

In [0]:
%python
df = spark.sql("select * from gizmobox.bronze.v_customers")

In [0]:
%python
df.describe().show()

In [0]:
%python
dbutils.data.summarize(df)

In [0]:
select customer_id,max(created_timestamp),max(customer_name),max(date_of_birth),max(email),max(member_since),max(telephone) from gizmobox.bronze.v_customers
where customer_id is not null
group by customer_id ;

-- this is not ideal as we are not handling the case where there is no customer name, date of birth, email, member since, telephone and also getting data from diffrent rows lets use CTE to solve this problem

In [0]:
select customer_id, max(t.created_timestamp ) from gizmobox.bronze.v_customers t 
group  by customer_id
-- we want to keep the newest record for each customer

In [0]:
-- CTE 
with cte_max as (select customer_id, max(t.created_timestamp ) tmax from gizmobox.bronze.v_customers t 
group  by customer_id) select distinct(*) from cte_max inner join gizmobox.bronze.v_customers t on cte_max.customer_id = t.customer_id and cte_max.tmax = t.created_timestamp order by 
t.customer_id

In [0]:
DROP VIEW v_customers_distinct

In [0]:
create or replace  view gizmobox.bronze.v_customers_distinct as (
select distinct(*) from gizmobox.bronze.v_customers where customer_id is not null order by customer_id )

In [0]:
select * from gizmobox.bronze.v_customers_distinct

In [0]:
-- CTE 
with cte_max as (select customer_id, max(t.created_timestamp ) tmax from gizmobox.bronze.v_customers t 
group  by customer_id) select * from cte_max inner join gizmobox.bronze.v_customers_distinct t on cte_max.customer_id = t.customer_id and cte_max.tmax = t.created_timestamp order by 
t.customer_id

Cast 

In [0]:
with cte_max as (select customer_id, max(t.created_timestamp ) tmax from gizmobox.bronze.v_customers t 
group  by customer_id) 
select CAST (tt.created_timestamp AS TIMESTAMP) as created_timestamp,
 tt.customer_id,
 tt.customer_name ,
CAST (tt.date_of_birth AS DATE) as date_of_birth,
 tt.email as email,
 tt.telephone as telephone,
CAST (tt.member_since AS DATE) as member_since
from gizmobox.bronze.v_customers_distinct tt INNER JOIN cte_max on cte_max.customer_id = tt.customer_id and cte_max.tmax = tt.created_timestamp order by tt.customer_id

In [0]:
create table gizmobox.silver.customers 
with cte_max as (select customer_id, max(t.created_timestamp ) tmax from gizmobox.bronze.v_customers t 
group  by customer_id) 
select CAST (tt.created_timestamp AS TIMESTAMP) as created_timestamp,
 tt.customer_id,
 tt.customer_name ,
CAST (tt.date_of_birth AS DATE) as date_of_birth,
 tt.email as email,
 tt.telephone as telephone,
CAST (tt.member_since AS DATE) as member_since
from gizmobox.bronze.v_customers_distinct tt INNER JOIN cte_max on cte_max.customer_id = tt.customer_id and cte_max.tmax = tt.created_timestamp order by tt.customer_id

In [0]:
select * from gizmobox.silver.customers

In [0]:
describe  extended gizmobox.silver.customers